# Nemotron-3-Nano-30B — v0.12 SFT Training on DGX Spark GB10

**Approach:** Format 4 SFT with per-expert MoE LoRA — 878M trainable params, 11,962 PEFT keys (186 base + 11,776 per-routed-expert)  
**Hardware:** NVIDIA DGX Spark GB10 — 130.7 GB HBM, Blackwell, aarch64  
**Key result:** **0.64 Kaggle score** — +0.06 vs v0.9 best (0.58) — driven by augmented data coverage and seq=4096 training

For Dockerfile, build instructions, and adapter key taxonomy, see the [v0.9 notebook](https://www.kaggle.com/code/gdataranger/nemotron-v0-9-sft-training-dgx-spark-gb10).  
For v0.14 (all 16 categories), see the [v0.14 notebook](https://www.kaggle.com/code/gdataranger/nemotron-v0-14-sft-training-dgx-spark-gb10).

## What changed vs v0.9

| | v0.9 (run13) | v0.12 (run14/run15) |
|---|---|---|
| Dataset | 13,730 rows | **25,500 rows** (v0.9 + 11,770 augmented) |
| Seq length | 2048 | **4096** |
| Warmstart | Fresh | **run13 step-1000** (then run14 step-300) |
| Learning rate | 2e-4 | **1e-4** (run14), **2e-4** (run15 retry) |
| Token filter | None | **≤ 4096 tokens** (15,502 of 25,500 used) |
| Kaggle Score | 0.58 | **0.64 ★** |

## Run history

| Run | Script | Steps | Seq | Warmstart | LR | Kaggle Score | Notes |
|---|---|---|---|---|---|---|---|
| **run14** | `run_train_v12.sh` | 300 (stopped) | 4096 | run13 step-1000 | 1e-4 | **0.64 ★** | Loss plateaued ~0.285 from step 80; stopped early |
| run15 | `run_train_v12.sh` | 200 (stopped) | 4096 | run14 step-300 | 2e-4 | 0.64 | LR raised; plateau 0.273–0.282; no improvement vs run14 |

## v0.12 dataset — augmented coverage

**Source**: [gdataranger/nemotron-v012-training-data](https://www.kaggle.com/datasets/gdataranger/nemotron-v012-training-data)

v0.9 had strong coverage of 6 categories but thin coverage of the remaining 10. v0.12 adds
11,770 augmented examples generated from `huikang/nemotron-master` deterministic solvers —
the same framework used by the 0.85+ leaderboard leaders.

After seq=4096 token filter: **15,502 of 25,500** examples used (9,998 dropped >4096 tokens).

Build the augmented examples:

```zsh
python scripts/generate_reasoner_data.py \
  --repo-root /tmp/huikang-repo/nemotron-master \
  --existing  data/v0.9_train.jsonl \
  --out       data/v0.12_augmented.jsonl

cat data/v0.9_train.jsonl data/v0.12_augmented.jsonl > data/v0.12_train.jsonl
```

## Format 4

All training examples use **Format 4** — no `<think>` opener in the assistant turn:

```
<|im_start|>assistant
{reasoning trace}
</think>
\boxed{answer}<|im_end|>
```

Nemotron-H's chat template (case 2: `</think>` present) leaves this token sequence intact during training.  
See [Format 4 investigation](https://github.com/msusol/kaggle-nemotron-model-reasoning-challenge/blob/main/docs/investigate/nemotron-chat-template.md).

In [ ]:
# ── RUN IDENTITY ──────────────────────────────────────────────────────────
# Set RUN_NAME before running any cells.
RUN_NAME = "v12_run14"   # or "v12_run15"

# ── DATA ─────────────────────────────────────────────────────────────────
TRAIN_FILE     = "data/v0.12_train.jsonl"   # 25,500 rows; 15,502 pass ≤4096 tok filter
MAX_SEQ_LENGTH = 4096

# ── INCREMENTAL TRAINING ──────────────────────────────────────────────────
# run14: warmstart from run13 step-1000
# run15: warmstart from run14 step-300
WARMSTART_ADAPTER = "output/adapter_v9_run13_ckpt"  # run15: "output/adapter_v12_spark_ckpt"

# ── TRAINING HYPERPARAMETERS ─────────────────────────────────────────────
MAX_STEPS     = 600       # run14: stopped at 300; run15: stopped at 200
LEARNING_RATE = 1e-4      # run14: 1e-4; run15: 2e-4
BATCH_SIZE    = 1
GRAD_ACCUM    = 16
LORA_R        = 32
LORA_ALPHA    = 32
CKPT_EVERY    = 100
SEED          = 3407

print(f"RUN_NAME:          {RUN_NAME}")
print(f"TRAIN_FILE:        {TRAIN_FILE}")
print(f"WARMSTART_ADAPTER: {WARMSTART_ADAPTER}")
print(f"MAX_SEQ_LENGTH:    {MAX_SEQ_LENGTH}")
print(f"MAX_STEPS:         {MAX_STEPS}")
print(f"LEARNING_RATE:     {LEARNING_RATE}")
print(f"CKPT_EVERY:        {CKPT_EVERY}")

## Training — v0.12 run14 (`run_train_v12.sh`)

run14 warmstarts from run13 step-1000 and trains on the v0.12 25,500-row augmented dataset.
Loss plateaued at ~0.285 from step 80 (LR=1e-4 too conservative for a deeply-converged warmstart).
Stopped at step 300 — Kaggle score **0.64 ★** (+0.06 vs run13 best).

```zsh
# Always in tmux — never run directly
tmux new -s train_v12

# run14: 600 steps planned, stopped at 300
RUN_NAME=v12_spark \
TRAIN_FILE=data/v0.12_train.jsonl \
WARMSTART_ADAPTER=output/adapter_v9_run13_ckpt \
MAX_SEQ_LENGTH=4096 \
MAX_STEPS=600 \
LEARNING_RATE=1e-4 \
bash scripts/run_train_v12.sh
```

**Verify warmstart engaged:**
```
[moe-lora] Warmstart: loaded 92 expert LoRA weights from output/adapter_v9_run13_ckpt/expert_lora_weights.pt
```

**Step time:** ~122 s/step → 300 steps ≈ 10h.

### Loss trajectory

| Step | Loss | Notes |
|---|---|---|
| 0 | — | warmstart from run13 (loss 0.122 at step 1000) |
| 80 | ~0.285 | plateau begins — LR=1e-4 insufficient for this warmstart |
| 300 | ~0.285 | stopped — submitted step-300 checkpoint |

## Training — v0.12 run15 (`run_train_v12.sh`)

run15 raises LR to 2e-4 with fresh Adam state, warmstarting from run14 step-300.
Loss plateaued 0.273–0.282 from step 50 with no clear descent. Stopped at step 200.

```zsh
tmux new -s train_v15

# run15: LR raised to 2e-4, warmstart from run14 step-300
RUN_NAME=v12_run15 \
TRAIN_FILE=data/v0.12_train.jsonl \
WARMSTART_ADAPTER=output/adapter_v12_spark_ckpt \
MAX_SEQ_LENGTH=4096 \
MAX_STEPS=200 \
LEARNING_RATE=2e-4 \
bash scripts/run_train_v12.sh
```

### Loss trajectory

| Step | Loss | Kaggle Score | Notes |
|---|---|---|---|
| 100 | 0.2738 | 0.64 | submitted |
| 200 | 0.2823 | 0.64 | run final — same score as run14 |

In [ ]:
import re, pathlib, subprocess

log_path = pathlib.Path(f"output/train_{RUN_NAME}.log")

if not log_path.exists():
    print(f"Log not found: {log_path}")
    print("Training runs on the DGX Spark host — logs are written there.")
    print("On Kaggle: not applicable.")
else:
    result = subprocess.run(["tail", "-20", str(log_path)], capture_output=True, text=True)
    print(result.stdout)

    # Parse step directly from tqdm bar: "| 100/600 [...] {'loss': '0.285', ...}"
    loss_pat = re.compile(r"\|\s+(\d+)/\d+\s+\[.*?\].*?'loss':\s+'([0-9.]+)'.*?'learning_rate':\s+'([0-9e.+-]+)'")
    ckpt_pat = re.compile(r"\[ckpt\] step (\d+)")
    losses, checkpoints = [], []
    with open(log_path) as f:
        for line in f:
            m = loss_pat.search(line)
            if m:
                losses.append((int(m.group(1)), float(m.group(2)), m.group(3)))
            c = ckpt_pat.search(line)
            if c:
                checkpoints.append(int(c.group(1)))

    if losses:
        print(f"\nLoss trajectory ({len(losses)} readings):")
        print(f"  {'step':>5}  {'loss':>7}  {'lr':>10}")
        for step, loss, lr in losses[-15:]:
            print(f"  {step:>5}  {loss:>7.4f}  {lr:>10}")
    if checkpoints:
        print(f"\nCheckpoints: {checkpoints}")
        print(f"Rolling checkpoint: output/adapter_{RUN_NAME}_ckpt/")

## Package and submit

Package from the **rolling checkpoint** immediately after each 100-step notification — it is overwritten at the next checkpoint.

```zsh
# run14 — package step-300 checkpoint
bash scripts/package_submission.sh \
  output/adapter_v12_spark_ckpt \
  /tmp/sub_v12_step300

kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f /tmp/sub_v12_step300/submission.zip \
  -m "v0.12 run14 step-300: warmstart run13, 25500 rows augmented, seq=4096"
```

```zsh
# run15 — package step-100 or step-200 checkpoint
bash scripts/package_submission.sh \
  output/adapter_v12_run15_step200 \
  /tmp/sub_v15_step200

kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f /tmp/sub_v15_step200/submission.zip \
  -m "v0.12 run15 step-200: lr=2e-4, warmstart run14, seq=4096"
```

## What's next — v0.14

v0.12 data covers 13 of 16 competition categories — `spelling`, `equation_numeric_deduce`,
and `equation_numeric_guess` are absent because all huikang traces for these categories
exceed 4,096 tokens. v0.14 adds short synthetic traces (≤ 500 tokens) for all 3, achieving
full 16-category coverage.

See the [v0.14 notebook](https://www.kaggle.com/code/gdataranger/nemotron-v0-14-sft-training-dgx-spark-gb10) for the data build pipeline and run17 training.